In [6]:
import duckdb

con = duckdb.connect()

con.sql("CREATE OR REPLACE VIEW customers AS SELECT * FROM '../data/raw/olist_customers_dataset.csv'")
con.sql("CREATE OR REPLACE VIEW geolocation AS SELECT * FROM '../data/raw/olist_geolocation_dataset.csv'")
con.sql("CREATE OR REPLACE VIEW order_items AS SELECT * FROM '../data/raw/olist_order_items_dataset.csv'")
con.sql("CREATE OR REPLACE VIEW payments AS SELECT * FROM '../data/raw/olist_order_payments_dataset.csv'")
con.sql("CREATE OR REPLACE VIEW reviews AS SELECT * FROM '../data/raw/olist_order_reviews_dataset.csv'")
con.sql("CREATE OR REPLACE VIEW orders AS SELECT * FROM '../data/raw/olist_orders_dataset.csv'")
con.sql("CREATE OR REPLACE VIEW products AS SELECT * FROM '../data/raw/olist_products_dataset.csv'")
con.sql("CREATE OR REPLACE VIEW sellers AS SELECT * FROM '../data/raw/olist_sellers_dataset.csv'")
con.sql("CREATE OR REPLACE VIEW category_name AS SELECT * FROM '../data/raw/product_category_name_translation.csv'")

con.sql("SHOW TABLES").show()



┌───────────────┐
│     name      │
│    varchar    │
├───────────────┤
│ category_name │
│ customers     │
│ geolocation   │
│ order_items   │
│ orders        │
│ payments      │
│ products      │
│ reviews       │
│ sellers       │
└───────────────┘



In [13]:
con.sql("""
    SELECT 
        order_id,
        payment_type,
        payment_installments,
        payment_value
    FROM payments
    WHERE payment_type = 'credit_card' and payment_value > 1000
    LIMIT 10
""")

order_idVARCHAR,payment_typeVARCHAR,payment_installmentsBIGINT,payment_valueDOUBLE
886b114d034f4ac1d39d964c1b2a8182,credit_card,10,2027.16
62d9b911d7c56cf455f660eecb8ddd3a,credit_card,10,1002.73
4ff8e28200e5a7a50b448cfaaf1f8ed3,credit_card,10,2288.31
ce6d150fb29ada17d2082f4847107665,credit_card,10,1586.47
e11fec6c25945565c1ef4f14fc3c03b7,credit_card,10,1995.69
360787554f41824600bdcba9687cd4f0,credit_card,3,1783.44
d1ea13fc1406243e6f6134582bf1bbc5,credit_card,10,1432.92
e41e5861711a10524b80a63d022ae776,credit_card,10,1180.51
264e91053cd726b067922356be6fd618,credit_card,2,1327.54
80d49171762a51f500bf6b774aa24617,credit_card,1,2787.87


In [27]:
con.sql("""
    SELECT 
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp,
        oi.price
    FROM orders o
    JOIN order_items oi
        ON o.order_id = oi.order_id
    WHERE o.order_purchase_timestamp BETWEEN '2017-12-15' AND '2017-12-22'
        AND o.order_status = 'delivered'
        AND oi.price BETWEEN 100 AND 200
    ORDER BY order_purchase_timestamp DESC
    LIMIT 10
""")

order_idVARCHAR,order_statusVARCHAR,order_purchase_timestampTIMESTAMP,priceDOUBLE
30f3a786f4600561beb44b9e2f3e727b,delivered,2017-12-21 23:30:00,149.9
b4411c4be4f9514e53fe707972f98685,delivered,2017-12-21 22:50:13,122.0
5f39964152b7c80a427cc4bffdfe757a,delivered,2017-12-21 21:41:31,119.99
a6832f84211588f24e426d167333044e,delivered,2017-12-21 21:27:00,119.9
21f4a37ec131158f1615d0f186f190af,delivered,2017-12-21 20:11:07,199.98
3ce5f19b0313072214db579286e3734c,delivered,2017-12-21 18:54:43,129.9
ed6b29fe01737a1e042a05988daf31db,delivered,2017-12-21 18:53:44,189.0
9a4efaf7ec75fc5669a3526c0a838e80,delivered,2017-12-21 18:37:24,149.0
6d1590646b6c320a92101bed4984f0e2,delivered,2017-12-21 17:11:28,113.9
1e1f2f647008bb63f8fb6cfe18d64467,delivered,2017-12-21 17:03:45,119.0


In [38]:
con.sql("""
    SELECT 
        customer_id,
        customer_city,
        customer_state
    FROM customers
    WHERE customer_city IN ('campinas', 'santos', 'sorocaba') OR LOWER(customer_city) LIKE '%sao%'
    LIMIT 15
""")

customer_idVARCHAR,customer_cityVARCHAR,customer_stateVARCHAR
18955e83d337fd6b2def6b18a428ac77,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,sao paulo,SP
4f2d8ab171c80ec8364f7c12e35b23ad,campinas,SP
fd826e7cf63160e536e0908c76c3f441,sao paulo,SP
eabebad39a88bb6f5b52376faec28612,sao paulo,SP
9b8ce803689b3562defaad4613ef426f,sao paulo,SP
2938121a40a20953c43caa8c98787fcb,sao paulo,SP
cb721d7b4f271fd87011c4c83462c076,sao paulo,SP
6e359a57a91f84095cc64e1b351aef8c,sao paulo,SP
e3109970a3fe8021d5ff82c577ce5606,sao paulo,SP


In [40]:
con.sql("""
    SELECT 
        COUNT(*) AS total_items,
        ROUND(SUM(price), 2) AS total_revenue,
        ROUND(AVG(price), 2) AS avg_price,
        ROUND(MIN(price), 2) AS min_price,
        ROUND(MAX(price), 2) AS max_price
    FROM order_items

""")

total_itemsBIGINT,total_revenueDOUBLE,avg_priceDOUBLE,min_priceDOUBLE,max_priceDOUBLE
112650,13591643.7,120.65,0.85,6735.0
